In [1]:
import pandas as pd
import vivarium_inputs
import vivarium.gbd_mapping as gbd_mapping
import pathlib
from lsff_utils import config_utils
from lsff_utils.results import expand_to_all_scenarios, aggregate_by_scenario

In [2]:
location = "india"
vehicle = "rice"

In [3]:
# Parameters
location = "nigeria"
vehicle = "rice"


In [4]:
scenarios = list(
    config_utils.get_location_fortificant_vehicle_intervention_scenarios()
    .pipe(lambda df: df[(df.location == location) & (df.vehicle == vehicle)])
    .intervention_scenario.unique()
) + ["zero", "baseline"]
scenarios

['intervention', 'zero', 'baseline']

In [5]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/person_time_anemia.parquet"
if pathlib.Path(path).is_file():
    pregnancy_person_time_anemia = pd.read_parquet(path)
else:
    pregnancy_person_time_anemia = expand_to_all_scenarios(
        pd.read_parquet(
            f"results/rescaled_pregnancy_results/rice/india/person_time_anemia.parquet"
        ).assign(value=0),
        scenarios,
    )
pregnancy_person_time_anemia

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,input_draw,random_seed,value
0,person_time,impairment,anemia,not_anemic,10_to_14,invalid,1,baseline,0,2,549.329082
1,person_time,impairment,anemia,not_anemic,10_to_14,invalid,2,baseline,0,2,693.987818
2,person_time,impairment,anemia,not_anemic,10_to_14,invalid,3,baseline,0,2,647.846670
3,person_time,impairment,anemia,not_anemic,10_to_14,invalid,4,baseline,0,2,727.658386
4,person_time,impairment,anemia,not_anemic,10_to_14,invalid,5,baseline,0,2,960.234717
...,...,...,...,...,...,...,...,...,...,...,...
53995,person_time,impairment,anemia,severe,95_plus,severe,1,baseline,0,9,0.000000
53996,person_time,impairment,anemia,severe,95_plus,severe,2,baseline,0,9,0.000000
53997,person_time,impairment,anemia,severe,95_plus,severe,3,baseline,0,9,0.000000
53998,person_time,impairment,anemia,severe,95_plus,severe,4,baseline,0,9,0.000000


In [6]:
pregnancy_person_time_anemia.groupby("scenario").random_seed.nunique()

scenario
baseline        10
intervention    10
zero            10
Name: random_seed, dtype: int64

In [7]:
pregnancy_person_time_anemia.sub_entity.value_counts()

not_anemic    13500
mild          13500
moderate      13500
severe        13500
Name: sub_entity, dtype: int64

In [8]:
total_pregnant_person_time = aggregate_by_scenario(pregnancy_person_time_anemia)
total_pregnant_person_time

scenario      wealth_quintile
baseline      1                  2.550640e+06
              2                  2.652239e+06
              3                  2.231159e+06
              4                  1.827720e+06
              5                  1.593579e+06
intervention  1                  2.550640e+06
              2                  2.652243e+06
              3                  2.231159e+06
              4                  1.827720e+06
              5                  1.593579e+06
zero          1                  2.550640e+06
              2                  2.652239e+06
              3                  2.231159e+06
              4                  1.827720e+06
              5                  1.593579e+06
Name: value, dtype: float64

In [9]:
anemic_pregnant_person_time = aggregate_by_scenario(
    pregnancy_person_time_anemia[
        pregnancy_person_time_anemia.sub_entity != "not_anemic"
    ]
)
anemic_pregnant_person_time

scenario      wealth_quintile
baseline      1                  1.348522e+06
              2                  1.452174e+06
              3                  1.106525e+06
              4                  8.028217e+05
              5                  6.592018e+05
intervention  1                  1.335079e+06
              2                  1.433284e+06
              3                  1.085183e+06
              4                  7.843708e+05
              5                  6.411456e+05
zero          1                  1.348522e+06
              2                  1.452174e+06
              3                  1.106525e+06
              4                  8.028217e+05
              5                  6.592018e+05
Name: value, dtype: float64

In [10]:
pregnant_anemia_prevalence_by_scenario = (
    anemic_pregnant_person_time / total_pregnant_person_time
).fillna(0)
pregnant_anemia_prevalence_by_scenario

scenario      wealth_quintile
baseline      1                  0.528700
              2                  0.547528
              3                  0.495942
              4                  0.439248
              5                  0.413661
intervention  1                  0.523429
              2                  0.540404
              3                  0.486376
              4                  0.429153
              5                  0.402331
zero          1                  0.528700
              2                  0.547528
              3                  0.495942
              4                  0.439248
              5                  0.413661
Name: value, dtype: float64

In [11]:
path = f"./results/{location}/{vehicle}/pregnant_anemia_prevalence_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
pregnant_anemia_prevalence_by_scenario.to_csv(path)

In [12]:
pop = pd.read_csv(f"../0100_data_prep/results/population/stratified/{location}.csv")
pop

,sex,age_start,age_end,pregnant,wealth_quintile,value
0,Female,0.0,0.019178,not_pregnant,1,17452.628925
1,Female,0.0,0.019178,not_pregnant,2,17380.997496
2,Female,0.0,0.019178,not_pregnant,3,16409.683914
3,Female,0.0,0.019178,not_pregnant,4,14424.706826
4,Female,0.0,0.019178,not_pregnant,5,13407.888483
...,...,...,...,...,...,...
280,Male,95.0,125.000000,not_pregnant,1,5056.112334
281,Male,95.0,125.000000,not_pregnant,2,4424.735951
282,Male,95.0,125.000000,not_pregnant,3,4548.244848
283,Male,95.0,125.000000,not_pregnant,4,4870.219344


In [13]:
pregnant_pop = pop[pop.pregnant == "pregnant"].groupby(["wealth_quintile"]).value.sum()
pregnant_pop

wealth_quintile
1    2.372743e+06
2    2.463522e+06
3    2.073741e+06
4    1.684268e+06
5    1.470164e+06
Name: value, dtype: float64

In [14]:
pregnancy_prevalent_anemia_cases_by_scenario = (
    pregnant_anemia_prevalence_by_scenario * pregnant_pop
)
pregnancy_prevalent_anemia_cases_by_scenario

scenario      wealth_quintile
baseline      1                  1.254469e+06
              2                  1.348846e+06
              3                  1.028454e+06
              4                  7.398108e+05
              5                  6.081498e+05
intervention  1                  1.241963e+06
              2                  1.331298e+06
              3                  1.008619e+06
              4                  7.228081e+05
              5                  5.914920e+05
zero          1                  1.254469e+06
              2                  1.348846e+06
              3                  1.028454e+06
              4                  7.398108e+05
              5                  6.081498e+05
Name: value, dtype: float64

In [15]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/transition_count_maternal_disorders.parquet"
if pathlib.Path(path).is_file():
    maternal_disorders_transition_counts = pd.read_parquet(path)
else:
    maternal_disorders_transition_counts = expand_to_all_scenarios(
        pd.read_parquet(
            f"results/rescaled_pregnancy_results/rice/india/transition_count_maternal_disorders.parquet"
        ).assign(value=0),
        scenarios,
    )

maternal_disorders_transition_counts

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,input_draw,random_seed,value
0,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,1,baseline,0,2,0.0
1,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,2,baseline,0,2,0.0
2,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,3,baseline,0,2,0.0
3,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,4,baseline,0,2,0.0
4,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,5,baseline,0,2,0.0
...,...,...,...,...,...,...,...,...,...,...,...
26995,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,1,baseline,0,9,0.0
26996,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,2,baseline,0,9,0.0
26997,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,3,baseline,0,9,0.0
26998,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,4,baseline,0,9,0.0


In [16]:
maternal_disorders_transition_counts.sub_entity.cat.categories

Index(['susceptible_to_maternal_disorders_to_maternal_disorders', 'maternal_disorders_to_recovered_from_maternal_disorders'], dtype='object')

In [17]:
maternal_disorders_incident_cases_by_scenario = aggregate_by_scenario(
    maternal_disorders_transition_counts[
        maternal_disorders_transition_counts.sub_entity
        == "susceptible_to_maternal_disorders_to_maternal_disorders"
    ]
)
maternal_disorders_incident_cases_by_scenario

scenario      wealth_quintile
baseline      1                  3.466947e+06
              2                  3.460960e+06
              3                  2.375109e+06
              4                  1.994907e+06
              5                  1.352214e+06
intervention  1                  3.462294e+06
              2                  3.452176e+06
              3                  2.362193e+06
              4                  1.985277e+06
              5                  1.342779e+06
zero          1                  3.466947e+06
              2                  3.460960e+06
              3                  2.375109e+06
              4                  1.994907e+06
              5                  1.352214e+06
Name: value, dtype: float64

In [18]:
path = (
    f"./results/{location}/{vehicle}/maternal_disorders_incident_cases_by_scenario.csv"
)
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
maternal_disorders_incident_cases_by_scenario.to_csv(path)

In [19]:
path = f"results/rescaled_child_results/{vehicle}/{location}/deaths.parquet"
if pathlib.Path(path).is_file():
    neonatal_deaths = pd.read_parquet(path).rename(
        columns={"maternal_scenario": "scenario"}
    )
else:
    neonatal_deaths = expand_to_all_scenarios(
        pd.read_parquet(f"results/rescaled_child_results/rice/india/deaths.parquet")
        .assign(value=0)
        .rename(columns={"maternal_scenario": "scenario"}),
        scenarios,
    )

neonatal_deaths

,measure,entity_type,entity,sub_entity,age_group,sex,wealth_quintile,child_scenario,scenario,input_draw,random_seed,value
0,deaths,cause,other_causes,other_causes,0_to_5_months,Female,1,baseline,intervention,0,2,4340.390854
1,deaths,cause,other_causes,other_causes,0_to_5_months,Female,2,baseline,intervention,0,2,4242.487301
2,deaths,cause,other_causes,other_causes,0_to_5_months,Female,3,baseline,intervention,0,2,3230.817252
3,deaths,cause,other_causes,other_causes,0_to_5_months,Female,4,baseline,intervention,0,2,2610.761416
4,deaths,cause,other_causes,other_causes,0_to_5_months,Female,5,baseline,intervention,0,2,2414.954310
...,...,...,...,...,...,...,...,...,...,...,...,...
1195,deaths,cause,other_causes,other_causes,18_to_59_months,Male,1,baseline,intervention,0,6,3426.624358
1196,deaths,cause,other_causes,other_causes,18_to_59_months,Male,2,baseline,intervention,0,6,3067.644664
1197,deaths,cause,other_causes,other_causes,18_to_59_months,Male,3,baseline,intervention,0,6,3035.010146
1198,deaths,cause,other_causes,other_causes,18_to_59_months,Male,4,baseline,intervention,0,6,2055.974615


In [20]:
neonatal_deaths_by_scenario = aggregate_by_scenario(neonatal_deaths)
neonatal_deaths_by_scenario

scenario      wealth_quintile
baseline      1                  181415.283877
              2                  185755.674730
              3                  155960.360073
              4                  126393.487040
              5                  109586.710426
intervention  1                  181382.649359
              2                  185723.040213
              3                  155960.360073
              4                  126328.218004
              5                  109554.075908
zero          1                  181415.283877
              2                  185755.674730
              3                  155960.360073
              4                  126393.487040
              5                  109586.710426
Name: value, dtype: float64

In [21]:
path = f"./results/{location}/{vehicle}/neonatal_deaths_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
neonatal_deaths_by_scenario.to_csv(path)

In [22]:
path = f"../0400_non_pregnant_anemia_model/results/{vehicle}/{location}/anemia_cases.parquet"
if pathlib.Path(path).is_file():
    non_pregnancy_anemia_cases = pd.read_parquet(path)
else:
    non_pregnancy_anemia_cases = expand_to_all_scenarios(
        pd.read_parquet(
            f"../0400_non_pregnant_anemia_model/results/rice/india/anemia_cases.parquet"
        ).assign(value=0),
        scenarios,
    )

non_pregnancy_anemia_cases

,sex,age_start,age_end,wealth_quintile,value,scenario
0,Female,0.0,0.019178,1,14769.329314,zero
1,Female,0.0,0.019178,2,14182.900679,zero
2,Female,0.0,0.019178,3,12601.737191,zero
3,Female,0.0,0.019178,4,10966.456900,zero
4,Female,0.0,0.019178,5,9094.030016,zero
...,...,...,...,...,...,...
745,Male,95.0,125.000000,1,3917.711147,intervention
746,Male,95.0,125.000000,2,3373.545142,intervention
747,Male,95.0,125.000000,3,3429.236719,intervention
748,Male,95.0,125.000000,4,3671.643364,intervention


In [23]:
non_pregnancy_prevalent_anemia_cases_by_scenario = aggregate_by_scenario(
    non_pregnancy_anemia_cases.assign(entity="anemia", input_draw="draw_0")
)
non_pregnancy_prevalent_anemia_cases_by_scenario

scenario      wealth_quintile
baseline      1                  2.614290e+07
              2                  2.354700e+07
              3                  2.153242e+07
              4                  2.088089e+07
              5                  1.685034e+07
intervention  1                  2.591710e+07
              2                  2.324122e+07
              3                  2.116498e+07
              4                  2.044900e+07
              5                  1.642205e+07
zero          1                  2.614290e+07
              2                  2.354700e+07
              3                  2.153242e+07
              4                  2.088089e+07
              5                  1.685034e+07
Name: value, dtype: float64

In [24]:
prevalent_anemia_cases_by_scenario = (
    pregnancy_prevalent_anemia_cases_by_scenario
    + non_pregnancy_prevalent_anemia_cases_by_scenario
)
prevalent_anemia_cases_by_scenario

scenario      wealth_quintile
baseline      1                  2.739737e+07
              2                  2.489585e+07
              3                  2.256088e+07
              4                  2.162070e+07
              5                  1.745849e+07
intervention  1                  2.715907e+07
              2                  2.457252e+07
              3                  2.217360e+07
              4                  2.117181e+07
              5                  1.701354e+07
zero          1                  2.739737e+07
              2                  2.489585e+07
              3                  2.256088e+07
              4                  2.162070e+07
              5                  1.745849e+07
Name: value, dtype: float64

In [25]:
path = f"./results/{location}/{vehicle}/prevalent_anemia_cases_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
prevalent_anemia_cases_by_scenario.to_csv(path)

In [26]:
path = f"../0500_neural_tube_defects_model/results/{location}/{vehicle}/ntd_cases_by_scenario.csv"
if pathlib.Path(path).is_file():
    ntd_cases_by_scenario = pd.read_csv(path)
else:
    ntd_cases_by_scenario = expand_to_all_scenarios(
        pd.read_csv(
            f"../0500_neural_tube_defects_model/results/india/rice/ntd_cases_by_scenario.csv"
        ).assign(value=0),
        scenarios,
    )

ntd_cases_by_scenario = ntd_cases_by_scenario.set_index(
    ["scenario", "wealth_quintile"]
).value
ntd_cases_by_scenario

scenario      wealth_quintile
zero          1                  7596.119668
              2                  7767.505148
              3                  7031.729552
              4                  6232.915620
              5                  5474.485622
baseline      1                  7596.119668
              2                  7767.505148
              3                  7031.729552
              4                  6232.915620
              5                  5474.485622
intervention  1                  7278.202366
              2                  7153.134515
              3                  6263.038217
              4                  5331.320975
              5                  4596.985698
Name: value, dtype: float64

In [27]:
path = f"./results/{location}/{vehicle}/ntd_cases_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
ntd_cases_by_scenario.to_csv(path)